# Consolidated 1000 epigenomes

"Universal annotation of the human genome through integration of over a thousand epigenomic datasets"
https://link.springer.com/article/10.1186/s13059-021-02572-z

See https://egg2.wustl.edu/roadmap/data/byFileType/metadata/EID_metadata.tab

Analysis is available in `epi_1000.sh`

## Compute

In [ ]:
import glob
import importlib
import os
import pickle
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from IPython.display import Image, display
from tqdm.auto import tqdm

# Load configuration
config_path = os.path.abspath(os.path.expanduser("~/work/omni-chromhmm/config.yaml"))
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

project_root = os.path.dirname(config_path)
scripts_dir = os.path.join(project_root, "scripts", "analysis")
scripts_rules_dir = os.path.join(project_root, "scripts", "rules")
workdir = os.path.expanduser(config.get("workdir", "."))

# Import the analysis methods directly (no CLI / subprocess).
sys.path.insert(0, scripts_dir)
sys.path.insert(0, scripts_rules_dir)

import analyze
import analyze_peaks
import compare
import compare_methods
import compare_inter_dataset as compare_out
import emission_similarity
import match
import summary_plots
from tqdm.auto import tqdm
import importlib
import compare
import match
import pickle
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm
from collections import defaultdict
from itertools import combinations

# Re-import the analysis modules so edits to scripts/analysis/*.py are picked up when this
# cell is re-run, without needing a kernel restart (plain `import` caches modules).
for _m in (analyze, analyze_peaks, compare, compare_methods,
           compare_out, emission_similarity, match, summary_plots):
    importlib.reload(_m)

In [ ]:
# Run everything relative to the pipeline working directory.
os.chdir(workdir)
print(f"Project root: {project_root}")
print(f"Scripts dir : {scripts_dir}")
print(f"Working dir : {workdir}")

# --- Parameters (mirrors the Snakefile) ----------------------------------
P = config["params"]
TOOLS = config["tools"]
DATASETS = config["datasets"]
MARKS = P["marks"]
CHROMHMM_BIN = P["chromhmm_bin"]
OMNI_BIN = P["omni_bin"]
HOMER_BIN = P["homer_bin"]
MACS2_BIN = P["macs2_bin"]
NSTATES = P["n_states"]
MATCH_METHOD = P.get("match_method", "comb")
CALLER_BIN = {"omni": OMNI_BIN, "homer": HOMER_BIN, "macs2": MACS2_BIN}

DO_REPLICATES = P.get("replicates", False)

# Peak callers to include. Edit to match the segmentations you actually produced;
# missing files are skipped gracefully throughout the notebook.
CALLERS = ["homer", "macs2", "omni"]

COORDS_DIR = os.path.join(workdir, TOOLS["coords_dir"])
GENCODE_GTF = os.path.join(workdir, TOOLS["gencode_gtf"])
MARKUPS_DIR = os.path.join(project_root, "markups")

# De-novo methods compared across datasets
INTER_DS_METHODS = (
        ["chromhmm_default"]
        + [f"kmeans_{c}" for c in CALLERS]
)
CHIP_DATASETS = [d for d in DATASETS if not d.endswith("_mint")]
MINT_DATASETS = [d for d in DATASETS if d.endswith("_mint")]
REP_DATASETS = [d for d in DATASETS if DO_REPLICATES and DATASETS[d].get("replicates")]


# --- Path helpers (mirror the Snakefile functions) -----------------------
def ds_of(folder):
    return folder.split("/")[0]


def folders_of(ds):
    fl = [ds]
    if DO_REPLICATES and DATASETS[ds].get("replicates"):
        fl += [f"{ds}/rep1", f"{ds}/rep2"]
    return fl


def ref_bed_path(ds):
    return f"{ds}/{DATASETS[ds]['ref_chromhmm']}_chromhmm.bed"


def seg_bin(path):
    for caller, size in CALLER_BIN.items():
        if f"/{caller}/" in path:
            return size
    return CHROMHMM_BIN


def inter_ds_bed(ds, method):
    cell = DATASETS[ds]["cell"]
    sfx = f"{MATCH_METHOD}_matched"
    if method == "chromhmm_default":
        return f"{ds}/chromhmm_default_result/{cell}_{NSTATES}_dense_{sfx}.bed"
    model, caller = method.split("_")  # kmeans , omni|homer|macs2
    return f"{ds}/{caller}/{caller}_kmeans_states_{sfx}.bed"


def existing(paths):
    """Keep only paths that exist on disk (skip segmentations not produced)."""
    return [p for p in paths if os.path.exists(p)]


print(f"Callers       : {CALLERS}")
print(f"Match variant : {MATCH_METHOD}")
print(f"Inter methods : {INTER_DS_METHODS}")


In [ ]:
EPI_1000_PATH = os.path.expanduser('~/data/2026_epi_1000')

In [ ]:
os.makedirs("out/epi_1000", exist_ok=True)

ref_path = os.path.expanduser("~/data/2026_omni_chromhmm/monocytes/ENCFF227EMB_chromhmm.bed")
ref_segs = match.load_bed(ref_path)
ref_colors = match.state_colors(ref_segs)

exxx_folders = sorted(
    [d for d in os.listdir(EPI_1000_PATH) if d.startswith("E") and os.path.isdir(os.path.join(EPI_1000_PATH, d))])

from functools import partial

methods = {
    "ChromHMM": "{f}_chromhmm/{f}_15_dense_matched.bed",
    "HOMER": "homer/{f}_homer_kmeans_states_matched.bed",
    "MACS2": "macs2/{f}_macs2_kmeans_states_matched.bed",
    "Omnipeak": "omni/{f}_omni_kmeans_states_matched.bed"
}

# Prepare tasks
tasks = []
for folder in exxx_folders:
    f_path = os.path.join(EPI_1000_PATH, folder)
    for method_name, pt in methods.items():
        p = os.path.join(f_path, pt.format(f=folder))
        if os.path.exists(p):
            tasks.append((folder, method_name, p, seg_bin(p)))

# Consolidate results using ProcessPoolExecutor with script functions
valid_tasks = [t for t in tasks if os.path.exists(t[2])]
paths = [t[2] for t in valid_tasks]
bins = [t[3] for t in valid_tasks]

print(f"Processing {len(valid_tasks)} existing segmentations...")
with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context('fork')) as executor:
    print("Loading BEDs...")
    all_segs = list(tqdm(executor.map(match.load_bed, paths), total=len(paths)))

print('Identify and filter out segmentations with non-matched states (missing names)')
matched_indices = [i for i, segs in enumerate(all_segs) if all('_' in s[3] for s in segs)]
if len(matched_indices) < len(all_segs):
    print(f"Filtering out {len(all_segs) - len(matched_indices)} non-matched segmentations.")
    for i in set(range(len(all_segs))) - set(matched_indices):
        folder, method_name, _, _ = valid_tasks[i]
        print(f"  - {folder} {method_name}")
    
    valid_tasks = [valid_tasks[i] for i in matched_indices]
    all_segs = [all_segs[i] for i in matched_indices]
    paths = [paths[i] for i in matched_indices]
    bins = [bins[i] for i in matched_indices]

all_segs = [s for s in all_segs]

In [ ]:
print('calculation of entropy or loading')
cache_tm = "out/epi_1000/all_tm.pkl"
if os.path.exists(cache_tm):
    print(f"Loading cached transition matrices from {cache_tm}...")
    with open(cache_tm, "rb") as f:
        all_tm_loaded, all_tm_noqh_loaded = pickle.load(f)
    if len(all_tm_loaded) == len(valid_tasks):
        all_tm, all_tm_noqh = all_tm_loaded, all_tm_noqh_loaded
    else:
        print(f"Cache size mismatch ({len(all_tm_loaded)} vs {len(valid_tasks)}), recomputing transition matrices...")
        os.remove(cache_tm) # Force recompute
        all_tm = None
else:
    all_tm = None

if all_tm is None:
    print("Calculating entropy...")
    with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context('fork')) as executor:
        all_tm = list(tqdm(executor.map(analyze.build_transition_matrix, all_segs, bins),
                           total=len(all_segs)))
        exclude_noqh = {"Quies", "Het", "15_Quies", "9_Het"}
        tm_noqh_func = partial(analyze.build_transition_matrix, exclude_states=exclude_noqh)
        all_tm_noqh = list(tqdm(executor.map(tm_noqh_func, all_segs, bins), total=len(all_segs)))
    print('Saving entropy information...')
    with open(cache_tm, "wb") as f:
        pickle.dump((all_tm, all_tm_noqh), f)

In [ ]:
print('statistics calculation or loading')
cache_stats = "out/epi_1000/all_stats.pkl"
if os.path.exists(cache_stats):
    print(f"Loading cached statistics from {cache_stats}...")
    with open(cache_stats, "rb") as f:
        all_stats_loaded = pickle.load(f)
    if len(all_stats_loaded) == len(valid_tasks):
        all_stats = all_stats_loaded
    else:
        print(f"Cache size mismatch ({len(all_stats_loaded)} vs {len(valid_tasks)}), recomputing statistics...")
        os.remove(cache_stats)
        all_stats = None
else:
    all_stats = None

if all_stats is None:
    print("Calculating statistics...")
    all_segs_5 = [s for s in all_segs]
    with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context('fork')) as executor:
        all_stats = list(tqdm(executor.map(compare.compute_segment_stats, all_segs_5), total=len(all_segs_5)))
    print('Saving statistics info...')
    with open(cache_stats, "wb") as f:
        pickle.dump(all_stats, f)

# Ensure all_segs is populated
all_segs = [s for s in all_segs]

In [ ]:
print('processing consolidated results')
res_path = "out/epi_1000/df_results.csv"
if os.path.exists(res_path):
    print(f"Loading results from {res_path}...")
    df_results_loaded = pd.read_csv(res_path)
    if len(df_results_loaded) == len(valid_tasks):
        df_results = df_results_loaded
    else:
        print(f"Cache size mismatch ({len(df_results_loaded)} vs {len(valid_tasks)}), recomputing results...")
        os.remove(res_path)
        df_results = None
else:
    df_results = None

if df_results is None:
    results = []
    for i in range(len(valid_tasks)):
        folder, method_name, _, _ = valid_tasks[i]
        states, counts, s_bp = all_tm[i]
        entropy, _, _, _ = analyze.transition_entropy(states, counts, s_bp)
        states_n, counts_n, s_bp_n = all_tm_noqh[i]
        entropy_noqh = analyze.transition_entropy(states_n, counts_n, s_bp_n)[0] if states_n else 0
        results.append({
            "Dataset": folder,
            "Method": method_name,
            "N_States": len(set(s[3] for s in all_segs[i])),
            "N_Segments": all_stats[i].get("n_segments", 0),
            "Entropy": entropy,
            "Entropy_NOQH": entropy_noqh
        })
    df_results = pd.DataFrame(results)
    print('Loading results info...')
    df_results.to_csv(res_path, index=False)

In [ ]:
# State composition
df_comp_path = "out/epi_1000/df_comp.csv"
if os.path.exists(df_comp_path):
    print('Loading state composition...')
    df_comp_loaded = pd.read_csv(df_comp_path)
    # df_comp has multiple rows per task, but we can check the number of unique (Dataset, Method) pairs
    if len(df_comp_loaded.groupby(["Dataset", "Method"])) == len(valid_tasks):
        df_comp = df_comp_loaded
    else:
        print(f"Cache size mismatch, recomputing state composition...")
        os.remove(df_comp_path)
        df_comp = None
else:
    df_comp = None

if df_comp is None:
    print('Computing state composition...')
    compositions = []
    for i in range(len(valid_tasks)):
        folder, method_name, _, _ = valid_tasks[i]
        segs = all_segs[i]
        w_lengths = match.state_lengths(segs)
        tot_bp = sum(w_lengths.values())
        for st, length in w_lengths.items():
            compositions.append({
                "Dataset": folder,
                "Method": method_name,
                "State": st,
                "Fraction": length / tot_bp if tot_bp > 0 else 0
            })
    df_comp = pd.DataFrame(compositions)
    print('Saving state composition info...')
    df_comp.to_csv(df_comp_path, index=False)

In [ ]:
# 7. Pairwise consistency (Jaccard and Kappa)
method_order = ["ChromHMM", "HOMER", "MACS2", "Omnipeak"]

# Helper for metric computation in the main process
def compute_pw_metrics(overlap, l1, l2):
    res = {}
    for mode in ["full", "noqh"]:
        excl = {"Quies", "Het", "15_Quies", "9_Het"} if mode == "noqh" else set()
        states = (set(l1.keys()) | set(l2.keys())) - excl
        if not states:
            res[mode] = (0.0, 0.0)
            continue
        total_overlap_area = sum(overlap.get((s1, s2), 0) for s1 in states for s2 in states)
        if total_overlap_area == 0:
            res[mode] = (0.0, 0.0)
            continue
        po = sum(overlap.get((s, s), 0) for s in states) / total_overlap_area
        a1_s = {s: sum(overlap.get((s, s2), 0) for s2 in states) for s in states}
        a2_s = {s: sum(overlap.get((s1, s), 0) for s1 in states) for s in states}
        pe = sum((a1_s[s] / total_overlap_area) * (a2_s[s] / total_overlap_area) for s in states)
        kappa = (po - pe) / (1 - pe) if pe < 1 else 1.0
        jaccards = [overlap.get((s, s), 0) / (a1_s[s] + a2_s[s] - overlap.get((s, s), 0))
                    for s in states if (a1_s[s] + a2_s[s] - overlap.get((s, s), 0)) > 0]
        res[mode] = (np.mean(jaccards) if jaccards else 0.0, kappa)
    return res

all_lengths = {}
df_pw_list = []

for m_name in method_order:
    cache_path = f"out/epi_1000/pw_cache_{m_name.lower()}.pkl"
    # ! rm {cache_path}
    if os.path.exists(cache_path):
        print(f"Loading cached pairwise consistency for {m_name} from {cache_path}...")
        with open(cache_path, "rb") as f:
            method_cache = pickle.load(f)
            df_pw_list.append(method_cache['df'])
            # Populate lengths just in case, although if we have df we don't strictly need them here
            if 'lengths' in method_cache:
                all_lengths.update(method_cache['lengths'])
            continue
    
    # Otherwise compute
    idxs = [idx for idx, task in enumerate(valid_tasks) if task[1] == m_name]
    missing_idxs = [i for i in idxs if i not in all_lengths]
    if missing_idxs:
        print(f"Computing lengths for {m_name} ({len(missing_idxs)} samples)...")
        segs_to_compute = [all_segs[i] for i in missing_idxs]
        with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context('fork')) as executor:
            new_lengths = list(tqdm(executor.map(match.state_lengths, segs_to_compute, chunksize=10), total=len(segs_to_compute)))
        for i, lengths in zip(missing_idxs, new_lengths):
            all_lengths[i] = lengths

    pairwise_tasks = [(m_name, i, j) for i, j in combinations(idxs, 2)]
    if len(pairwise_tasks) > 1000:
        random.seed(42)
        pairwise_tasks = random.sample(pairwise_tasks, 1000)
        pairwise_tasks.sort()
    
    if not pairwise_tasks:
        print(f"No pairwise tasks for {m_name}, skipping.")
        continue

    print(f"Computing pairwise overlaps for {m_name} ({len(pairwise_tasks)} pairs)...")
    # Prepare segment data for this method to avoid redundant list creation in the map
    local_segs_5 = {i: all_segs[i] for i in idxs}
    with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context('fork')) as executor:
        overlaps = list(tqdm(executor.map(match.pair_overlap,
                                          [local_segs_5[i] for m, i, j in pairwise_tasks],
                                          [local_segs_5[j] for m, i, j in pairwise_tasks],
                                          chunksize=100),
                             total=len(pairwise_tasks)))
    
    pw_data = []
    for (m_name, i, j), overlap in zip(pairwise_tasks, overlaps):
        metrics = compute_pw_metrics(overlap, all_lengths[i], all_lengths[j])
        for mode in ["full", "noqh"]:
            jaccard, kappa = metrics[mode]
            pw_data.append({"Method": m_name, "Mode": mode.upper(), "Jaccard": jaccard, "Kappa": kappa})
    
    df_method = pd.DataFrame(pw_data)
    print(f"Saving updated cache to {cache_path}...")
    method_lengths = {i: all_lengths[i] for i in idxs}
    with open(cache_path, "wb") as f:
        pickle.dump({'overlaps': overlaps, 'df': df_method, 'lengths': method_lengths}, f)
    df_pw_list.append(df_method)

# Final aggregation
if df_pw_list:
    df_pw = pd.concat(df_pw_list, ignore_index=True)
    # Ensure correct order for plotting
    df_pw['Method'] = pd.Categorical(df_pw['Method'], categories=method_order, ordered=True)
    df_pw = df_pw.sort_values(['Method', 'Mode']).reset_index(drop=True)
else:
    df_pw = pd.DataFrame(columns=["Method", "Mode", "Jaccard", "Kappa"])


## Plotting

In [ ]:
method_palette = {
    "ChromHMM": summary_plots.BIN_COLORS["default"],
    "HOMER": summary_plots.BIN_COLORS["homer"],
    "MACS2": summary_plots.BIN_COLORS["macs2"],
    "Omnipeak": summary_plots.BIN_COLORS["omnipeak"]
}

In [ ]:
# 1. Number of states
fig, ax = plt.subplots(figsize=(6, 4.2))
sns.barplot(data=df_results, x="Method", y="N_States", hue="Method", palette=method_palette, order=list(methods.keys()),
            capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0}, dodge=False, ax=ax)
ax.set_title("Number of unique matched states per method", fontsize=11, fontweight="bold")
ax.set_ylabel("Number of states", fontsize=9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.grid(axis="y", alpha=0.3)
# Add labels over error bars
method_order = list(methods.keys())
yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
for i, method in enumerate(method_order):
    m = df_results[df_results["Method"] == method]["N_States"].mean()
    s = df_results[df_results["Method"] == method]["N_States"].sem()
    if pd.isna(m): continue
    top = m + (s if not pd.isna(s) else 0)
    ax.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=6)
fig.tight_layout()
fig.savefig("out/epi_1000/n_states.png", bbox_inches="tight")
plt.close(fig)

In [ ]:
# 2. Number of segments
fig, ax = plt.subplots(figsize=(6, 4.2))
# Divide by 1000 for consistency with analysis.ipynb
df_results_copy = df_results.copy()
df_results_copy["N_Segments_K"] = df_results_copy["N_Segments"] / 1000.0
sns.barplot(data=df_results_copy, x="Method", y="N_Segments_K", hue="Method", palette=method_palette,
            order=list(methods.keys()), capsize=0.05,
            errorbar="se", err_kws={"linewidth": 2.0}, dodge=False, ax=ax)
ax.set_title("Number of segments per method", fontsize=11, fontweight="bold")
ax.set_ylabel("Segments (×10³)", fontsize=9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.grid(axis="y", alpha=0.3)
# Add labels over error bars
method_order = list(methods.keys())
yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
for i, method in enumerate(method_order):
    m = df_results_copy[df_results_copy["Method"] == method]["N_Segments_K"].mean()
    s = df_results_copy[df_results_copy["Method"] == method]["N_Segments_K"].sem()
    if pd.isna(m): continue
    top = m + (s if not pd.isna(s) else 0)
    ax.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=6)
fig.tight_layout()
fig.savefig("out/epi_1000/n_segments.png", bbox_inches="tight")
plt.close(fig)

In [ ]:
# 3. States composition (average per type)
# Use natural sort for states, important for names like '1_TssA', '10_TssB', '2_TssAFlnk'
def sort_states_natural(states):
    return sorted(states, key=lambda x: (summary_plots.STATE_IDX.get(x, 999), 
                                         int(x.split('_')[0]) if '_' in x and x.split('_')[0].isdigit() else 999,
                                         x))

states_order = sort_states_natural(df_comp['State'].unique())
BREAK_LOW = 0.20
BREAK_HIGH = 0.40
figw = max(12, len(states_order) * len(method_palette) * 0.22)

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, sharex=True,
    figsize=(figw, 6),
    gridspec_kw={"height_ratios": [1, 4], "hspace": 0.06},
)

for ax in (ax_top, ax_bot):
    sns.barplot(
        data=df_comp, x="State", y="Fraction", hue="Method",
        order=states_order, hue_order=list(methods.keys()), palette=method_palette,
        ax=ax, capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0},
        legend=(ax is ax_top)
    )

ax_top.set_ylim(BREAK_HIGH, 1.02)
ax_bot.set_ylim(0, BREAK_LOW)

# Hide the inner spines to create the visual break
ax_top.spines["bottom"].set_visible(False)
ax_bot.spines["top"].set_visible(False)
ax_top.tick_params(axis="x", bottom=False)

# Draw diagonal break marks
d = 0.012
kwargs = dict(transform=fig.transFigure, color="k", clip_on=False, linewidth=0.8)
for ax, sign in [(ax_top, -1), (ax_bot, 1)]:
    x0, x1 = ax.get_position().x0, ax.get_position().x1
    y = ax.get_position().y0 if sign == 1 else ax.get_position().y1
    for x in (x0, x1):
        fig.add_artist(plt.Line2D([x - d, x + d], [y + sign * d * 1.5, y - sign * d * 1.5], **kwargs))

for ax in (ax_top, ax_bot):
    ax.grid(axis="y", alpha=0.3, linewidth=0.5)
    ax.tick_params(axis="y", labelsize=8)

ax_bot.tick_params(axis="x", labelsize=8, rotation=45)
ax_bot.set_xlabel("Chromatin state", fontsize=9)
ax_top.set_xlabel("")
ax_top.set_title("Average state composition per method", fontsize=10, fontweight="bold")

ax_top.legend(title="Method", fontsize=8, title_fontsize=9,
              bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)
if ax_bot.get_legend():
    ax_bot.get_legend().remove()

ax_top.set_ylabel("")
ax_bot.set_ylabel("Fraction of genome", fontsize=9)

plt.savefig("out/epi_1000/avg_composition.png", bbox_inches='tight')
plt.close(fig)

In [ ]:
# 4. States composition for each method individually
# Collect state colors from all loaded segmentations for better matching
state_colors_hex = {}
for i in range(len(all_segs)):
    for row in all_segs[i]:
        name = row[3]
        color = row[4] if len(row) > 4 else "0,0,0"
        if name not in state_colors_hex and color != "0,0,0":
            state_colors_hex[name] = analyze.rgb_str_to_hex(color)

# Fallback to ref_colors and summary_plots.STATE_COLORS
for s in states_order:
    if s not in state_colors_hex or state_colors_hex[s] == "#000000":
        # Try exact match in ref_colors
        if s in ref_colors:
            state_colors_hex[s] = analyze.rgb_str_to_hex(ref_colors[s])
        else:
            # Try prefix match in summary_plots.STATE_COLORS
            name_part = s.split('_')[1] if '_' in s else s
            found_color = None
            for canonical, rgb in summary_plots.STATE_COLORS.items():
                if name_part.startswith(canonical):
                    found_color = '#{:02x}{:02x}{:02x}'.format(int(rgb[0]*255), int(rgb[1]*255), int(rgb[2]*255))
                    break
            if found_color:
                state_colors_hex[s] = found_color
            else:
                state_colors_hex[s] = analyze.rgb_str_to_hex(ref_colors.get(s, "128,128,128"))

for method in methods.keys():
    method_df = df_comp[df_comp["Method"] == method]
    if method_df.empty: continue
    pivot_df = method_df.pivot(index="Dataset", columns="State", values="Fraction").fillna(0)
    avail_states = [s for s in states_order if s in pivot_df.columns]
    pivot_df = pivot_df[avail_states]
    colors = [state_colors_hex.get(s, "#888888") for s in avail_states]

    ax = pivot_df.plot(kind='bar', stacked=True, figsize=(20, 8), color=colors, width=0.8)
    ax.set_title(f"State composition per dataset - {method}", fontsize=11, fontweight="bold")
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize='small', title="State")
    ax.set_xlabel("Dataset (EID)", fontsize=9)
    ax.set_ylabel("Fraction of Genome", fontsize=9)
    if len(pivot_df) > 50:
        ax.tick_params(axis='x', labelsize=6)
    fig = ax.get_figure()
    fig.tight_layout()
    fig.savefig(f"out/epi_1000/composition_{method.lower()}.png", bbox_inches="tight")
    plt.close(fig)

In [ ]:
# 5. Transition matrix Entropy
fig, ax = plt.subplots(figsize=(6, 4.2))
sns.barplot(data=df_results, x="Method", y="Entropy", hue="Method", palette=method_palette, order=list(methods.keys()),
            capsize=0.05,
            errorbar="se", err_kws={"linewidth": 2.0}, dodge=False, ax=ax)
ax.set_title("Transition matrix entropy (full)", fontsize=11, fontweight="bold")
ax.set_ylabel("Entropy (bits)", fontsize=9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.grid(axis="y", alpha=0.3)
# Add labels over error bars
method_order = list(methods.keys())
yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
for i, method in enumerate(method_order):
    m = df_results[df_results["Method"] == method]["Entropy"].mean()
    s = df_results[df_results["Method"] == method]["Entropy"].sem()
    if pd.isna(m): continue
    top = m + (s if not pd.isna(s) else 0)
    ax.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=6)
fig.tight_layout()
fig.savefig("out/epi_1000/entropy.png", bbox_inches="tight")
plt.close(fig)

# 5b. Transition matrix Entropy (NOQH)
fig, ax = plt.subplots(figsize=(6, 4.2))
sns.barplot(data=df_results, x="Method", y="Entropy_NOQH", hue="Method", palette=method_palette,
            order=list(methods.keys()), capsize=0.05,
            errorbar="se", err_kws={"linewidth": 2.0}, dodge=False, ax=ax)
ax.set_title("Transition matrix entropy (NOQH, Excl. Quies/Het)", fontsize=11, fontweight="bold")
ax.set_ylabel("Entropy (bits)", fontsize=9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.grid(axis="y", alpha=0.3)
# Add labels over error bars
method_order = list(methods.keys())
yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
for i, method in enumerate(method_order):
    m = df_results[df_results["Method"] == method]["Entropy_NOQH"].mean()
    s = df_results[df_results["Method"] == method]["Entropy_NOQH"].sem()
    if pd.isna(m): continue
    top = m + (s if not pd.isna(s) else 0)
    ax.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=6)
fig.tight_layout()
fig.savefig("out/epi_1000/entropy_noqh.png", bbox_inches="tight")
plt.close(fig)

In [ ]:
# 6. Summary average state composition per method (stacked)
avg_comp = df_comp.groupby(['Method', 'State'])['Fraction'].mean().reset_index()
pivot_avg = avg_comp.pivot(index='Method', columns='State', values='Fraction').fillna(0)
# Reorder by methods keys to keep consistent order
pivot_avg = pivot_avg.reindex(list(methods.keys()))
avail_states = [s for s in states_order if s in pivot_avg.columns]
pivot_avg = pivot_avg[avail_states]
colors = [state_colors_hex.get(s, "#888888") for s in avail_states]

plt.figure(figsize=(8, 5))
ax = pivot_avg.plot(kind='bar', stacked=True, color=colors, width=0.6, ax=plt.gca())
ax.set_title("Average state composition per method", fontsize=11, fontweight="bold")
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize='small', title="State")
ax.set_xlabel("Method", fontsize=9)
ax.set_ylabel("Average Fraction of Genome", fontsize=9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.grid(axis="y", alpha=0.3, linewidth=0.5)
plt.tight_layout()
plt.savefig("out/epi_1000/avg_composition_stacked.png", bbox_inches="tight")
plt.close()

In [ ]:
# Plotting
for mode in ["FULL", "NOQH"]:
    for metric in ["Jaccard", "Kappa"]:
        fig, ax = plt.subplots(figsize=(6, 4.5))
        df_mode = df_pw[df_pw["Mode"] == mode]
        sns.barplot(data=df_mode, x="Method", y=metric, hue="Method", palette=method_palette,
                    order=list(methods.keys()),
                    capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0}, dodge=False, ax=ax)
        ax.set_title(f"Pairwise {metric} consistency ({mode})", fontsize=11, fontweight="bold")
        ax.set_ylabel(f"{metric} index", fontsize=9)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
        ax.grid(axis="y", alpha=0.3)

        # Add labels over error bars
        method_order = list(methods.keys())
        yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
        for i, method in enumerate(method_order):
            subset = df_mode[df_mode["Method"] == method][metric]
            if subset.empty: continue
            m, s = subset.mean(), subset.sem()
            top = m + (s if not pd.isna(s) else 0)
            ax.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=6)

        fig.tight_layout()
        fig.savefig(f"out/epi_1000/pairwise_consistency_{mode.lower()}_{metric.lower()}.png", bbox_inches="tight")
        plt.close(fig)

## Results

In [ ]:
display(Image(filename="out/epi_1000/n_states.png"))
display(Image(filename="out/epi_1000/n_segments.png"))
display(Image(filename="out/epi_1000/avg_composition.png"))
display(Image(filename="out/epi_1000/avg_composition_stacked.png"))
display(Image(filename="out/epi_1000/entropy.png"))
display(Image(filename="out/epi_1000/entropy_noqh.png"))
display(Image(filename="out/epi_1000/pairwise_consistency_full_jaccard.png"))
display(Image(filename="out/epi_1000/pairwise_consistency_full_kappa.png"))
display(Image(filename="out/epi_1000/pairwise_consistency_noqh_jaccard.png"))
display(Image(filename="out/epi_1000/pairwise_consistency_noqh_kappa.png"))
for method in methods.keys():
    path = f"out/epi_1000/composition_{method.lower()}.png"
    if os.path.exists(path):
        display(Image(filename=path))

# Reference 18-state core K27ac segmentations

### 1. Identify files and compute/load stats

In [ ]:
# 1. Identify files
epi_18_core_files = sorted(glob.glob(f"{EPI_1000_PATH}/E*_18_core_K27ac_dense.bed.gz"))
epi_18_core_ids = [os.path.basename(f).split("_")[0] for f in epi_18_core_files]

import json

def compute_pw_metrics_18(overlap, l1, l2):
    res = {}
    for mode in ["full", "noqh"]:
        excl = {"18_Quies", "13_Het"} if mode == "noqh" else set()
        states = (set(l1.keys()) | set(l2.keys())) - excl
        if not states:
            res[mode] = (0.0, 0.0)
            continue
        total_overlap_area = sum(overlap.get((s1, s2), 0) for s1 in states for s2 in states)
        if total_overlap_area == 0:
            res[mode] = (0.0, 0.0)
            continue
        po = sum(overlap.get((s, s), 0) for s in states) / total_overlap_area
        a1_s = {s: sum(overlap.get((s, s2), 0) for s2 in states) for s in states}
        a2_s = {s: sum(overlap.get((s1, s), 0) for s1 in states) for s in states}
        pe = sum((a1_s[s] / total_overlap_area) * (a2_s[s] / total_overlap_area) for s in states)
        kappa = (po - pe) / (1 - pe) if pe < 1 else 1.0
        jaccards = [overlap.get((s, s), 0) / (a1_s[s] + a2_s[s] - overlap.get((s, s), 0))
                    for s in states if (a1_s[s] + a2_s[s] - overlap.get((s, s), 0)) > 0]
        res[mode] = (np.mean(jaccards) if jaccards else 0.0, kappa)
    return res

def get_file_stats_18(f, segs):
    eid = os.path.basename(f).split("_")[0]
    n_segments = len(segs)
    lengths = match.state_lengths(segs)
    total = sum(lengths.values())
    composition = [{"State": s, "Fraction": l / total if total > 0 else 0} for s, l in lengths.items()]
    entropy = {}
    segs4 = [(r[0], r[1], r[2], r[3]) for r in segs]
    for mode in ["full", "noqh"]:
        excl = {"18_Quies", "13_Het"} if mode == "noqh" else set()
        states, counts, state_bp = analyze.build_transition_matrix(segs4, 200, exclude_states=excl)
        entropy[mode] = analyze.transition_entropy(states, counts, state_bp)[0] if states else 0
    colors = {row[3]: analyze.rgb_str_to_hex(row[4] if len(row) > 4 else "0,0,0") for row in segs}
    return eid, {"n_segments": n_segments, "composition": composition, "entropy": entropy, "colors": colors, "lengths": lengths}

# 2. Computation
results_18 = []
comp_18 = []
entropy_18 = []
state_colors_18 = {}
loaded_segs_18 = {}

print(f"Processing {len(epi_18_core_files)} 18-state files...")
for f, eid in tqdm(zip(epi_18_core_files, epi_18_core_ids), total=len(epi_18_core_files)):
    segs = match.load_bed(f)
    loaded_segs_18[eid] = segs
    eid, stats = get_file_stats_18(f, segs)
    
    results_18.append({"Dataset": eid, "N_Segments": stats["n_segments"]})
    for c in stats["composition"]:
        comp_18.append({"Dataset": eid, "State": c["State"], "Fraction": c["Fraction"]})
    for m, val in stats["entropy"].items():
        entropy_18.append({"Dataset": eid, "Mode": m.upper(), "Entropy": val})
    state_colors_18.update(stats["colors"])

df_segments_18 = pd.DataFrame(results_18)
df_comp_18 = pd.DataFrame(comp_18)
df_entropy_18 = pd.DataFrame(entropy_18)

# Save results
df_segments_18.to_csv("out/epi_1000/df_segments_18.csv", index=False)
df_comp_18.to_csv("out/epi_1000/df_comp_18.csv", index=False)
df_entropy_18.to_csv("out/epi_1000/df_entropy_18.csv", index=False)
with open("out/epi_1000/state_colors_18.json", "w") as f:
    json.dump(state_colors_18, f)

# 3. Pairwise metrics
if os.path.exists("out/epi_1000/df_pw_18.csv"):
    df_pw_18 = pd.read_csv("out/epi_1000/df_pw_18.csv")
else:
    print("Computing pairwise overlaps for all pairs...")
    loaded_lengths = {eid: match.state_lengths(segs) for eid, segs in loaded_segs_18.items()}
    pairs = [tuple(sorted((id1, id2))) for i, id1 in enumerate(epi_18_core_ids) for id2 in epi_18_core_ids[i+1:]]
    if len(pairs) > 1000:
        random.seed(42)
        pairs = random.sample(pairs, 1000)
        pairs.sort()
    
    pw_data_18 = []
    for id1, id2 in tqdm(pairs):
        overlap = match.pair_overlap(loaded_segs_18[id1], loaded_segs_18[id2])
        metrics = compute_pw_metrics_18(overlap, loaded_lengths[id1], loaded_lengths[id2])
        for mode, (jaccard, kappa) in metrics.items():
            pw_data_18.append({"Dataset1": id1, "Dataset2": id2, "Mode": mode.upper(), "Jaccard": jaccard, "Kappa": kappa})
    df_pw_18 = pd.DataFrame(pw_data_18)
    df_pw_18.to_csv("out/epi_1000/df_pw_18.csv", index=False)


### 2. Plotting

In [ ]:
# 1) Segments number distribution
plt.figure(figsize=(3, 4))
ax = plt.gca()
sns.barplot(x=["18-core"] * len(df_segments_18), y=df_segments_18["N_Segments"], color='skyblue', capsize=0.1, errorbar="se", ax=ax)
ax.set_title("Distribution of segment numbers (18-state core K27ac)", fontsize=11, fontweight="bold")
ax.set_ylabel("Number of segments", fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Add label over error bar
m = df_segments_18["N_Segments"].mean()
s = df_segments_18["N_Segments"].sem()
yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
ax.text(0, m + s + 0.01 * yrange, f"{m:.0f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("out/epi_1000/epi_18_core_segments_dist.png", bbox_inches="tight")
plt.close()


# Custom sort for 18-state names (e.g., 1_TssA, 10_TssB, 2_TssAFlnk)
def sort_states_18(states):
    return sorted(states, key=lambda x: int(x.split('_')[0]) if '_' in x and x.split('_')[0].isdigit() else 999)


# Sort states and prepare colors
all_states_18 = sort_states_18(df_comp_18['State'].unique())

# Fallback to summary_plots.STATE_COLORS if some colors are missing or black
for s in all_states_18:
    if s not in state_colors_18 or state_colors_18[s] == "#000000":
        # Try prefix match in summary_plots.STATE_COLORS
        name_part = s.split('_')[1] if '_' in s else s
        found_color = None
        for canonical, rgb in summary_plots.STATE_COLORS.items():
            if name_part.startswith(canonical):
                found_color = '#{:02x}{:02x}{:02x}'.format(int(rgb[0]*255), int(rgb[1]*255), int(rgb[2]*255))
                break
        if found_color:
            state_colors_18[s] = found_color
        elif s not in state_colors_18:
            state_colors_18[s] = "#888888"

# Create color list for pandas stacked bar plot (maintains order)
colors_18 = [state_colors_18.get(s, "#888888") for s in all_states_18]

# 2) State composition per dataset
pivot_comp_18 = df_comp_18.pivot(index='Dataset', columns='State', values='Fraction').fillna(0)
pivot_comp_18 = pivot_comp_18[all_states_18]  # Reorder columns
plt.figure(figsize=(15, 6))
ax = pivot_comp_18.plot(kind='bar', stacked=True, ax=plt.gca(), width=0.8, color=colors_18)
ax.set_title("State composition per dataset (18-state core K27ac)", fontsize=11, fontweight="bold")
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize='x-small', title="State")
ax.set_xlabel("Dataset", fontsize=9)
ax.set_ylabel("Fraction of Genome", fontsize=9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=6)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("out/epi_1000/epi_18_core_composition_per_dataset.png", bbox_inches="tight")
plt.close()

# 3) Average composition
avg_comp_18 = df_comp_18.groupby('State')['Fraction'].mean().reindex(all_states_18)
plt.figure(figsize=(10, 5))
# Use sns.barplot with dictionary palette for robust color mapping
ax = sns.barplot(x=avg_comp_18.index, y=avg_comp_18.values, hue=avg_comp_18.index, palette=state_colors_18,
                 legend=False)
ax.set_title("Average state composition (18-state core K27ac)", fontsize=11, fontweight="bold")
ax.set_ylabel("Average Fraction of Genome", fontsize=9)
ax.set_xlabel("State", fontsize=9)
ax.set_xticklabels(all_states_18, rotation=45, ha="right", fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("out/epi_1000/epi_18_core_avg_composition.png", bbox_inches="tight")
plt.close()

# 4) Pairwise consistency plots
for mode in ["FULL", "NOQH"]:
    for metric in ["Jaccard", "Kappa"]:
        fig, ax = plt.subplots(figsize=(3, 4))
        df_mode = df_pw_18[df_pw_18["Mode"] == mode]
        sns.barplot(x=["18-core"] * len(df_mode), y=df_mode[metric], color='lightgreen', capsize=0.1, errorbar="se",
                    ax=ax)
        ax.set_title(f"Pairwise {metric} consistency ({mode}) - 18-core", fontsize=11, fontweight="bold")
        ax.set_ylabel(f"{metric} index", fontsize=9)
        ax.grid(axis="y", alpha=0.3)

        # Add label over error bar
        m = df_mode[metric].mean()
        s = df_mode[metric].sem()
        yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
        ax.text(0, m + s + 0.01 * yrange, f"{m:.2f}", ha="center", va="bottom", fontsize=8)

        fig.tight_layout()
        fig.savefig(f"out/epi_1000/epi_18_core_pw_{mode.lower()}_{metric.lower()}.png", bbox_inches="tight")
        plt.close(fig)

# 5) Transition entropy plots
for mode in ["FULL", "NOQH"]:
    plt.figure(figsize=(3, 4))
    ax = plt.gca()
    df_mode = df_entropy_18[df_entropy_18["Mode"] == mode]
    sns.barplot(x=["18-core"] * len(df_mode), y=df_mode["Entropy"], color='lightcoral', capsize=0.1, errorbar="se", ax=ax)
    mode_label = "(Full)" if mode == "FULL" else "(Excl. Quies/Het)"
    ax.set_title(f"Transition Entropy {mode_label}\n(18-state core K27ac)", fontsize=11, fontweight="bold")
    ax.set_ylabel("Entropy (bits)", fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    # Add label over error bar
    m = df_mode["Entropy"].mean()
    s = df_mode["Entropy"].sem()
    yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
    ax.text(0, m + s + 0.01 * yrange, f"{m:.3f}", ha="center", va="bottom", fontsize=8)

    plt.tight_layout()
    plt.savefig(f"out/epi_1000/epi_18_core_entropy_{mode.lower()}.png", bbox_inches="tight")
    plt.close()


### 3. Show images

In [ ]:
display(Image(filename="out/epi_1000/epi_18_core_segments_dist.png"))
display(Image(filename="out/epi_1000/epi_18_core_composition_per_dataset.png"))
display(Image(filename="out/epi_1000/epi_18_core_avg_composition.png"))
for mode in ["FULL", "NOQH"]:
    for metric in ["Jaccard", "Kappa"]:
        display(Image(filename=f"out/epi_1000/epi_18_core_pw_{mode.lower()}_{metric.lower()}.png"))
for mode in ["full", "noqh"]:
    display(Image(filename=f"out/epi_1000/epi_18_core_entropy_{mode.lower()}.png"))

# 15 states core models joint reordered vs inidividual matched

### 1. Identify files and compute/load stats

In [ ]:
# 1. Identify files
joint_15_files = sorted(glob.glob(f"{EPI_1000_PATH}/E*_15_coreMarks_dense_joint_reodered.bed.gz"))
indiv_15_files = sorted(glob.glob(f"{EPI_1000_PATH}/E*_15_coreMarks_dense_matched.bed"))

joint_15_ids_map = {os.path.basename(f).split("_")[0]: f for f in joint_15_files}
indiv_15_ids_map = {os.path.basename(f).split("_")[0]: f for f in indiv_15_files}

common_15_ids = sorted(list(set(joint_15_ids_map.keys()) & set(indiv_15_ids_map.keys())))
print(f"Found {len(common_15_ids)} common samples for 15-state models")

def compute_pw_metrics_15(overlap, l1, l2):
    res = {}
    for mode in ["full", "noqh"]:
        # Standard ChromHMM 15-state: 9 is Het, 15 is Quies
        excl = {"15_Quies", "9_Het"} if mode == "noqh" else set()
        states = (set(l1.keys()) | set(l2.keys())) - excl
        if not states:
            res[mode] = (0.0, 0.0)
            continue
        total_overlap_area = sum(overlap.get((s1, s2), 0) for s1 in states for s2 in states)
        if total_overlap_area == 0:
            res[mode] = (0.0, 0.0)
            continue
        po = sum(overlap.get((s, s), 0) for s in states) / total_overlap_area
        a1_s = {s: sum(overlap.get((s, s2), 0) for s2 in states) for s in states}
        a2_s = {s: sum(overlap.get((s1, s), 0) for s1 in states) for s in states}
        pe = sum((a1_s[s] / total_overlap_area) * (a2_s[s] / total_overlap_area) for s in states)
        kappa = (po - pe) / (1 - pe) if pe < 1 else 1.0
        jaccards = [overlap.get((s, s), 0) / (a1_s[s] + a2_s[s] - overlap.get((s, s), 0))
                    for s in states if (a1_s[s] + a2_s[s] - overlap.get((s, s), 0)) > 0]
        res[mode] = (np.mean(jaccards) if jaccards else 0.0, kappa)
    return res

def get_file_stats_15_raw(f, segs):
    eid = os.path.basename(f).split("_")[0]
    n_segments = len(segs)
    lengths = match.state_lengths(segs)
    total = sum(lengths.values())
    composition = [{"State": s, "Fraction": l / total if total > 0 else 0} for s, l in lengths.items()]
    colors = {row[3]: analyze.rgb_str_to_hex(row[4] if len(row) > 4 else "0,0,0") for row in segs}
    return {"Dataset": eid, "n_segments": n_segments, "composition": composition, "colors": colors}

# 2. Computation / Loading
cache_15_path = "out/epi_1000/stats_15_cache.pkl"
# ! rm {cache_15_path}
if os.path.exists(cache_15_path):
    with open(cache_15_path, "rb") as f_cache:
        cache_15 = pickle.load(f_cache)
else:
    cache_15 = {}

results_15 = []
updated_15 = False

print(f"Computing stats for 15-state models...")
for eid in tqdm(common_15_ids):
    for t_name, f_map in [("Joint", joint_15_ids_map), ("Individual", indiv_15_ids_map)]:
        key = (eid, t_name)
        if key in cache_15:
            stats = cache_15[key]
        else:
            f_path = f_map[eid]
            segs = match.load_bed(f_path)
            stats = get_file_stats_15_raw(f_path, segs)
            stats["Type"] = t_name
            cache_15[key] = stats
            updated_15 = True
        results_15.append(stats)

if updated_15:
    os.makedirs(os.path.dirname(cache_15_path), exist_ok=True)
    with open(cache_15_path, "wb") as f_cache:
        pickle.dump(cache_15, f_cache)

df_segments_15 = pd.DataFrame([{"Dataset": r["Dataset"], "Type": r["Type"], "N_Segments": r["n_segments"]} for r in results_15])
df_comp_15 = pd.DataFrame([{"Dataset": r["Dataset"], "Type": r["Type"], "State": c["State"], "Fraction": c["Fraction"]} 
                            for r in results_15 for c in r["composition"]])
state_colors_15 = {}
for r in results_15:
    state_colors_15.update(r["colors"])

# Save aggregated results
df_segments_15.to_csv("out/epi_1000/df_segments_15.csv", index=False)
df_comp_15.to_csv("out/epi_1000/df_comp_15.csv", index=False)
with open("out/epi_1000/state_colors_15.json", "w") as f:
    json.dump(state_colors_15, f)


In [ ]:
# 3. Pairwise metrics computation
df_15_states_path = "out/epi_1000/df_pw_15.csv"
cache_pw_15_path = "out/epi_1000/pw_15_cache.pkl"

# ! rm {df_15_states_path}
# ! rm {cache_pw_15_path}
if os.path.exists(cache_pw_15_path):
    with open(cache_pw_15_path, "rb") as f_cache:
        cache_pw_15 = pickle.load(f_cache)
else:
    cache_pw_15 = {}

pw_data_15 = []
updated_pw_15 = False

for t_name, f_map in [("Individual", indiv_15_ids_map), ("Joint", joint_15_ids_map)]:
    print(f"Checking pairwise overlaps for 15-state {t_name}...")
    current_ids = common_15_ids
    pairs = [tuple(sorted((id1, id2))) for i, id1 in enumerate(current_ids) for id2 in current_ids[i+1:]]
    if len(pairs) > 1000:
        random.seed(42)
        pairs = random.sample(pairs, 1000)
        pairs.sort()
    
    last_id1 = None
    s1, l1 = None, None
    for id1, id2 in tqdm(pairs, desc=f"Overlaps {t_name}"):
        key = (t_name, id1, id2)
        if key in cache_pw_15:
            metrics = cache_pw_15[key]
        else:
            if id1 != last_id1:
                s1 = match.load_bed(f_map[id1])
                l1 = match.state_lengths(s1)
                last_id1 = id1
            s2 = match.load_bed(f_map[id2])
            l2 = match.state_lengths(s2)
            overlap = match.pair_overlap(s1, s2)
            metrics = compute_pw_metrics_15(overlap, l1, l2)
            cache_pw_15[key] = metrics
            updated_pw_15 = True
        
        for mode, (jaccard, kappa) in metrics.items():
            pw_data_15.append({"Type": t_name, "Dataset1": id1, "Dataset2": id2, 
                               "Mode": mode.upper(), "Jaccard": jaccard, "Kappa": kappa})

if updated_pw_15:
    os.makedirs(os.path.dirname(cache_pw_15_path), exist_ok=True)
    with open(cache_pw_15_path, "wb") as f_cache:
        pickle.dump(cache_pw_15, f_cache)

df_pw_15 = pd.DataFrame(pw_data_15)
df_pw_15.to_csv(df_15_states_path, index=False)


### 2. Plotting 15-state models

In [ ]:
# 1) Segments number distribution
plt.figure(figsize=(4, 4))
ax = plt.gca()
sns.barplot(data=df_segments_15, x="Type", y="N_Segments", order=["Individual", "Joint"], palette={"Joint": "skyblue", "Individual": "lightcoral"}, capsize=0.1, errorbar="se", ax=ax)
ax.set_title("Distribution of segment numbers (15-state)", fontsize=11, fontweight="bold")
ax.set_ylabel("Number of segments", fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Add labels over error bars
for i, t in enumerate(["Individual", "Joint"]):
    subset = df_segments_15[df_segments_15["Type"] == t]
    if not subset.empty:
        m = subset["N_Segments"].mean()
        s = subset["N_Segments"].sem()
        yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
        ax.text(i, m + s + 0.01 * yrange, f"{m:.0f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("out/epi_1000/epi_15_segments_comparison.png", bbox_inches="tight")
plt.close()

In [ ]:
# Custom sort for 15-state names
def sort_states_15(states):
    return sorted(states, key=lambda x: int(x.split('_')[0]) if '_' in x and x.split('_')[0].isdigit() else 999)

all_states_15 = sort_states_15(df_comp_15['State'].unique())

# 2) Average state composition for both types on a single plot
plt.figure(figsize=(12, 6))
ax = plt.gca()
sns.barplot(data=df_comp_15, x="State", y="Fraction", hue="Type", order=all_states_15,
            hue_order=["Individual", "Joint"], palette={"Joint": "skyblue", "Individual": "lightcoral"},
            errorbar=None, ax=ax)
ax.set_title("Average state composition comparison (15-state)", fontsize=11, fontweight="bold")
ax.set_ylabel("Average Fraction of Genome", fontsize=9)
ax.set_xlabel("State", fontsize=9)
ax.set_xticklabels(all_states_15, rotation=45, ha="right", fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("out/epi_1000/epi_15_avg_composition_comparison.png", bbox_inches="tight")
plt.close()

In [ ]:
# 3) For each type state composition for each sample
# Fallback to summary_plots.STATE_COLORS if some colors are missing or black
for s in all_states_15:
    if s not in state_colors_15 or state_colors_15[s] == "#000000":
        # Try prefix match in summary_plots.STATE_COLORS
        name_part = s.split('_')[1] if '_' in s else s
        found_color = None
        for canonical, rgb in summary_plots.STATE_COLORS.items():
            if name_part.startswith(canonical):
                found_color = '#{:02x}{:02x}{:02x}'.format(int(rgb[0]*255), int(rgb[1]*255), int(rgb[2]*255))
                break
        if found_color:
            state_colors_15[s] = found_color
        elif s not in state_colors_15:
            state_colors_15[s] = "#888888"

colors_15_list = [state_colors_15.get(s, "#888888") for s in all_states_15]

for t in ["Individual", "Joint"]:
    pivot_comp = df_comp_15[df_comp_15["Type"] == t].pivot(index='Dataset', columns='State', values='Fraction').fillna(0)
    pivot_comp = pivot_comp.reindex(columns=all_states_15)
    plt.figure(figsize=(15, 6))
    ax = pivot_comp.plot(kind='bar', stacked=True, ax=plt.gca(), width=0.8, color=colors_15_list)
    ax.set_title(f"State composition per dataset ({t} 15-state)", fontsize=11, fontweight="bold")
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize='x-small', title="State")
    ax.set_xlabel("Dataset", fontsize=9)
    ax.set_ylabel("Fraction of Genome", fontsize=9)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=6)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"out/epi_1000/epi_15_{t.lower()}_composition_per_dataset.png", bbox_inches="tight")
    plt.close()

In [ ]:
# 4) Pairwise consistency plots
for mode in ["FULL", "NOQH"]:
    for metric in ["Jaccard", "Kappa"]:
        plt.figure(figsize=(4, 4))
        ax = plt.gca()
        sns.barplot(data=df_pw_15[df_pw_15["Mode"] == mode], x="Type", y=metric, 
                    order=["Individual", "Joint"], palette={"Joint": "skyblue", "Individual": "lightcoral"},
                    capsize=0.1, errorbar="se", ax=ax)
        ax.set_title(f"Pairwise {metric} consistency ({mode})\n(15-state models)", fontsize=11, fontweight="bold")
        ax.set_ylabel(f"{metric} index", fontsize=9)
        ax.grid(axis='y', alpha=0.3)
        
        # Add labels over error bars
        for i, t in enumerate(["Individual", "Joint"]):
            subset = df_pw_15[(df_pw_15["Type"] == t) & (df_pw_15["Mode"] == mode)]
            if not subset.empty:
                m = subset[metric].mean()
                s = subset[metric].sem()
                yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
                ax.text(i, m + s + 0.01 * yrange, f"{m:.2f}", ha="center", va="bottom", fontsize=8)
        
        plt.tight_layout()
        plt.savefig(f"out/epi_1000/epi_15_pw_{mode.lower()}_{metric.lower()}.png", bbox_inches="tight")
        plt.close()


### 3. Show 15-state analysis images

In [ ]:
display(Image(filename="out/epi_1000/epi_15_segments_comparison.png"))
display(Image(filename="out/epi_1000/epi_15_avg_composition_comparison.png"))
display(Image(filename="out/epi_1000/epi_15_individual_composition_per_dataset.png"))
display(Image(filename="out/epi_1000/epi_15_joint_composition_per_dataset.png"))
for mode in ["FULL", "NOQH"]:
    for metric in ["Jaccard", "Kappa"]:
        display(Image(filename=f"out/epi_1000/epi_15_pw_{mode.lower()}_{metric.lower()}.png"))


# Cell type differences in chromatin

### 1. Compute

In [ ]:
os.makedirs("out/epi_1000/consistency", exist_ok=True)

# 1. De-novo methods
method_segs = defaultdict(list)
for task, segs in zip(valid_tasks, all_segs):
    method_name = task[1]
    method_segs[method_name].append(segs)

denovo_counts = {}
denovo_dists = {}
for method_name in ["ChromHMM", "HOMER", "MACS2", "Omnipeak"]:
    if method_name not in method_segs: continue
    cache_path = f"out/epi_1000/consistency/denovo_{method_name.lower()}.pkl"
    dist_cache_path = f"out/epi_1000/consistency/denovo_{method_name.lower()}_dist.pkl"
    # ! rm {cache_path} {dist_cache_path}
    if os.path.exists(cache_path):
        print(f"Loading cached consistency for de-novo {method_name}...")
        with open(cache_path, "rb") as f:
            denovo_counts[method_name] = pickle.load(f)
    else:
        print(f"--- Analyzing de-novo {method_name} ---")
        segs_list = method_segs[method_name]
        counts = analyze.compute_state_consistency(segs_list, show_progress=True)
        with open(cache_path, "wb") as f:
            pickle.dump(counts, f)
        denovo_counts[method_name] = counts

    if os.path.exists(dist_cache_path):
        print(f"Loading cached dist consistency for de-novo {method_name}...")
        with open(dist_cache_path, "rb") as f:
            denovo_dists[method_name] = pickle.load(f)
    else:
        print(f"--- Analyzing de-novo {method_name} dist consistency ---")
        segs_list = method_segs[method_name]
        # Use pre-computed lengths for speed
        lengths_list = [match.state_lengths(s) for s in tqdm(segs_list, desc="Lengths", leave=False)]
        dists = []
        for i, j in tqdm(combinations(range(len(lengths_list)), 2), 
                         total=len(lengths_list)*(len(lengths_list)-1)//2, 
                         desc="Distances", leave=False):
            w1, w2 = lengths_list[i], lengths_list[j]
            # Cosine similarity of state distributions
            states = sorted(set(w1.keys()) | set(w2.keys()), key=analyze._natural_sort_key)
            v1 = np.array([w1.get(s, 0) for s in states])
            v2 = np.array([w2.get(s, 0) for s in states])
            sim = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)) if np.linalg.norm(v1) > 0 and np.linalg.norm(v2) > 0 else 0
            dists.append(sim)
        with open(dist_cache_path, "wb") as f:
            pickle.dump(dists, f)
        denovo_dists[method_name] = dists

In [ ]:
# 2. 15-state models
state_15_counts = {}
state_15_dists = {}
for t_name, f_map in [("Individual", indiv_15_ids_map), ("Joint", joint_15_ids_map)]:
    cache_path = f"out/epi_1000/consistency/15state_{t_name.lower()}.pkl"
    dist_cache_path = f"out/epi_1000/consistency/15state_{t_name.lower()}_dist.pkl"
    # ! rm {cache_path} {dist_cache_path}
    if os.path.exists(cache_path):
        print(f"Loading cached consistency for 15-state {t_name}...")
        with open(cache_path, "rb") as f:
            state_15_counts[t_name] = pickle.load(f)
    else:
        print(f"--- Analyzing 15-state {t_name} ---")
        files = [f_map[eid] for eid in common_15_ids]
        segs_list = [match.load_bed(f) for f in tqdm(files, desc=f"Loading {t_name}")]
        counts = analyze.compute_state_consistency(segs_list, show_progress=True)
        with open(cache_path, "wb") as f:
            pickle.dump(counts, f)
        state_15_counts[t_name] = counts

    if os.path.exists(dist_cache_path):
        print(f"Loading cached dist consistency for 15-state {t_name}...")
        with open(dist_cache_path, "rb") as f:
            state_15_dists[t_name] = pickle.load(f)
    else:
        print(f"--- Analyzing 15-state {t_name} dist consistency ---")
        files = [f_map[eid] for eid in common_15_ids]
        segs_list = [match.load_bed(f) for f in tqdm(files, desc=f"Loading {t_name}")]
        lengths_list = [match.state_lengths(s) for s in tqdm(segs_list, desc="Lengths", leave=False)]
        dists = []
        for i, j in tqdm(combinations(range(len(lengths_list)), 2), 
                         total=len(lengths_list)*(len(lengths_list)-1)//2, 
                         desc="Distances", leave=False):
            w1, w2 = lengths_list[i], lengths_list[j]
            states = sorted(set(w1.keys()) | set(w2.keys()), key=analyze._natural_sort_key)
            v1 = np.array([w1.get(s, 0) for s in states])
            v2 = np.array([w2.get(s, 0) for s in states])
            sim = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)) if np.linalg.norm(v1) > 0 and np.linalg.norm(v2) > 0 else 0
            dists.append(sim)
        with open(dist_cache_path, "wb") as f:
            pickle.dump(dists, f)
        state_15_dists[t_name] = dists

In [ ]:
# 3. 18-state models
cache_path_18 = "out/epi_1000/consistency/18state_core.pkl"
dist_cache_path_18 = "out/epi_1000/consistency/18state_core_dist.pkl"
if os.path.exists(cache_path_18):
    print("Loading cached consistency for 18-state Core...")
    with open(cache_path_18, "rb") as f:
        counts_18 = pickle.load(f)
else:
    print("--- Analyzing 18-state Core ---")
    segs_18_list = [loaded_segs_18[eid] for eid in epi_18_core_ids]
    counts_18 = analyze.compute_state_consistency(segs_18_list, show_progress=True)
    with open(cache_path_18, "wb") as f:
        pickle.dump(counts_18, f)

if os.path.exists(dist_cache_path_18):
    print("Loading cached dist consistency for 18-state Core...")
    with open(dist_cache_path_18, "rb") as f:
        dists_18 = pickle.load(f)
else:
    print("--- Analyzing 18-state Core dist consistency ---")
    segs_18_list = [loaded_segs_18[eid] for eid in epi_18_core_ids]
    lengths_list = [match.state_lengths(s) for s in tqdm(segs_18_list, desc="Lengths", leave=False)]
    dists = []
    for i, j in tqdm(combinations(range(len(lengths_list)), 2), 
                     total=len(lengths_list)*(len(lengths_list)-1)//2, 
                     desc="Distances", leave=False):
        w1, w2 = lengths_list[i], lengths_list[j]
        states = sorted(set(w1.keys()) | set(w2.keys()), key=analyze._natural_sort_key)
        v1 = np.array([w1.get(s, 0) for s in states])
        v2 = np.array([w2.get(s, 0) for s in states])
        sim = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)) if np.linalg.norm(v1) > 0 and np.linalg.norm(v2) > 0 else 0
        dists.append(sim)
    with open(dist_cache_path_18, "wb") as f:
        pickle.dump(dists, f)
    dists_18 = dists

### 2. Plotting

In [ ]:
# 1. De-novo methods
# Replace white with black for visibility in consistency plots

for method_name, counts in denovo_counts.items():
    print(f"--- Plotting de-novo {method_name} ---")
    M = len(method_segs[method_name])
    for state in sorted(counts.keys(), key=analyze._natural_sort_key):
        print(f"  {state}: {counts[state][M]}")
    # Use actual colors from segmentations
    colors = match.state_colors(method_segs[method_name][0])
    analyze.plot_state_consistency(counts, f"De-novo {method_name}", f"out/epi_1000/epi_denovo_{method_name.lower()}_consistency.png", colors=colors)

# 2. 15-state models
for t_name, counts in state_15_counts.items():
    print(f"--- Plotting 15-state {t_name} ---")
    M = len(common_15_ids)
    for state in sorted(counts.keys(), key=analyze._natural_sort_key):
        print(f"  {state}: {counts[state][M]}")
    analyze.plot_state_consistency(counts, f"15-state {t_name}", f"out/epi_1000/epi_15_{t_name.lower()}_consistency.png", colors=state_colors_15)

# 3. 18-state models
print("--- Plotting 18-state Core ---")
M_18 = len(epi_18_core_ids)
for state in sorted(counts_18.keys(), key=analyze._natural_sort_key):
    print(f"  {state}: {counts_18[state][M_18]}")
analyze.plot_state_consistency(counts_18, "18-state Core", "out/epi_1000/epi_18_core_consistency.png", colors=state_colors_18)

# 4. Distribution consistency (composition similarity)
for method_name, dists in denovo_dists.items():
    plt.figure(figsize=(5, 3))
    sns.histplot(dists, kde=True, bins=50)
    plt.title(f"Composition Similarity — De-novo {method_name}")
    plt.xlabel("Cosine similarity")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(f"out/epi_1000/epi_denovo_{method_name.lower()}_dist.png")
    plt.close()

for t_name, dists in state_15_dists.items():
    plt.figure(figsize=(5, 3))
    sns.histplot(dists, kde=True, bins=50)
    plt.title(f"Composition Similarity — 15-state {t_name}")
    plt.xlabel("Cosine similarity")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(f"out/epi_1000/epi_15_{t_name.lower()}_dist.png")
    plt.close()

plt.figure(figsize=(5, 3))
sns.histplot(dists_18, kde=True, bins=50)
plt.title("Composition Similarity — 18-state Core")
plt.xlabel("Cosine similarity")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("out/epi_1000/epi_18_core_dist_consistency.png")
plt.close()

### 3. Display

In [ ]:
for method_name in ["ChromHMM", "HOMER", "MACS2", "Omnipeak"]:
    if method_name in denovo_counts:
        display(Image(filename=f"out/epi_1000/epi_denovo_{method_name.lower()}_consistency.png"))
        display(Image(filename=f"out/epi_1000/epi_denovo_{method_name.lower()}_dist.png"))
for t_name in ["Individual", "Joint"]:
    if t_name in state_15_counts:
        display(Image(filename=f"out/epi_1000/epi_15_{t_name.lower()}_consistency.png"))
        display(Image(filename=f"out/epi_1000/epi_15_{t_name.lower()}_dist.png"))
display(Image(filename="out/epi_1000/epi_18_core_consistency.png"))
display(Image(filename="out/epi_1000/epi_18_core_dist_consistency.png"))
